In [1]:
import torch

# multinomial

In [15]:
# 概率分布
w = torch.tensor([4, 7, 2, 3], dtype=torch.float)
w

tensor([4., 7., 2., 3.])

In [16]:
# 无放回随机抽取
torch.multinomial(w, num_samples=3)

tensor([3, 2, 1])

In [21]:
# 有放回
torch.multinomial(w,
                  num_samples=5,
                  replacement=True)

tensor([2, 0, 1, 1, 3])

In [38]:
# 概率分布
w = torch.randint(0, 10, (3, 4),
                  dtype=torch.float)
w

tensor([[6., 9., 9., 7.],
        [8., 1., 5., 8.],
        [3., 6., 8., 8.]])

In [40]:
torch.multinomial(w, num_samples=2)

tensor([[1, 3],
        [0, 3],
        [1, 3]])

# gather

In [45]:
a = torch.arange(0, 12).reshape(3, 4)
a

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

In [ ]:
# 在指定维度抽取多个元素
torch.gather(a, dim=1,
             index=torch.tensor([[0, 3]]))

tensor([[0, 3]])

In [ ]:
# 在指定维度抽取多个元素
torch.gather(a, dim=1,
             index=torch.tensor([[0, 3], [2, 1]]))

tensor([[0, 3],
        [6, 5]])

# Top-p sampling

In [3]:
from llama.generation import sample_top_p

In [ ]:
_ = torch.manual_seed(3)
# 预测下一个Token的分布
# (N, V)
probs = torch.rand((2, 5))
# 建议归一化
probs = torch.softmax(probs, dim=-1)
probs

tensor([[0.1653, 0.1829, 0.2190, 0.1691, 0.2637],
        [0.1110, 0.2261, 0.2198, 0.1893, 0.2539]])

In [88]:
sample_top_p(probs, p=0.9)

tensor([[2],
        [4]])

In [84]:
def top_p_sample(probs: torch.Tensor, top_p: float) -> torch.Tensor:
    """Top-p 采样"""
    # 从大到小排序，同时返回索引
    probs_sort, probs_idx = torch.sort(probs, dim=-1, descending=True)
    # 计算概率累加值
    probs_sum = torch.cumsum(probs_sort, dim=-1)
    # 只留累加值>top_p的前几个概率，其余太小的概率直接置零
    # 这里不能直接probs_sum>top_p，这样会少一位
    probs_sort[probs_sum - probs_sort > top_p] = 0.0

    # 归一化概率
    # 有时候传入的probs就是归一化过的，此处便多余
    probs_sort.div_(probs_sort.sum(dim=-1, keepdim=True))

    # 随机抽取一个，当然是概率越大的越容易抽到
    next_token = torch.multinomial(probs_sort, num_samples=1)
    next_token = torch.gather(probs_idx, dim=-1, index=next_token)
    return next_token

In [95]:
top_p_sample(probs, top_p=0.9)

tensor([[2],
        [4]])

In [76]:
# 从大到小排序，同时返回索引
probs_sort, probs_idx = torch.sort(probs, dim=-1, descending=True)
probs_sort
probs_idx

tensor([[0.2637, 0.2190, 0.1829, 0.1691, 0.1653],
        [0.2539, 0.2261, 0.2198, 0.1893, 0.1110]])

tensor([[4, 2, 1, 3, 0],
        [4, 1, 2, 3, 0]])

In [77]:
# 累加
probs_sum = torch.cumsum(probs_sort, dim=-1)
probs_sum

tensor([[0.2637, 0.4828, 0.6657, 0.8347, 1.0000],
        [0.2539, 0.4799, 0.6997, 0.8890, 1.0000]])

In [78]:
p = 0.5
mask = probs_sum - probs_sort
mask
mask = mask > p
# 概率太小的直接被置零
probs_sort[mask] = 0.0
probs_sort

tensor([[0.0000, 0.2637, 0.4828, 0.6657, 0.8347],
        [0.0000, 0.2539, 0.4799, 0.6997, 0.8890]])

tensor([[0.2637, 0.2190, 0.1829, 0.0000, 0.0000],
        [0.2539, 0.2261, 0.2198, 0.0000, 0.0000]])

In [79]:
probs_sort.div_(probs_sort.sum(dim=-1, keepdim=True))
probs_sort

tensor([[0.3962, 0.3290, 0.2748, 0.0000, 0.0000],
        [0.3628, 0.3231, 0.3141, 0.0000, 0.0000]])

tensor([[0.3962, 0.3290, 0.2748, 0.0000, 0.0000],
        [0.3628, 0.3231, 0.3141, 0.0000, 0.0000]])

In [80]:
next_token = torch.multinomial(probs_sort, num_samples=1)
next_token

tensor([[0],
        [2]])

In [81]:
next_token = torch.gather(probs_idx, -1, next_token)
next_token

tensor([[4],
        [2]])